### Map reduce to Summarize Large documents

In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

api_key=os.getenv("GROQ_API_KEY")
llm=ChatGroq(groq_api_key=api_key,model="openai/gpt-oss-20b")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.13', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7e12599b6cf0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7e12599b7770>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("I Have a Dream by Martin Luther King Jr.pdf")
docs = loader.load_and_split()
docs

/tmp/ipykernel_16030/862298826.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/home/anas/LEARNING/GEN-AI/LangChain/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2013-01-16T16:57:22-05:00', 'author': 'Peter', 'moddate': '2014-09-29T14:17:47-04:00', 'title': 'Microsoft Word - The Most Dangerous Game', 'source': 'I Have a Dream by Martin Luther King Jr.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='I\xa0HAVE\xa0A\xa0DREAM\xa0\xa0\nMartin\xa0Luther\xa0King,\xa0Jr.\xa0\n\xa0\nI\xa0am\xa0happy\xa0to\xa0join\xa0with\xa0you\xa0today\xa0in\xa0what\xa0will\xa0go\xa0down\xa0in\xa0history\xa0as\xa0the\xa0greatest\xa0demonstration\xa0for\xa0freedom\xa0in\xa0the\xa0\nhistory\xa0of\xa0our\xa0nation.\xa0\n\xa0\nFive\xa0score\xa0years\xa0ago,\xa0a\xa0great\xa0American,\xa0in\xa0whose\xa0symbolic\xa0shadow\xa0we\xa0stand\xa0today,\xa0signed\xa0the\xa0Emancipation\xa0\nProclamation.\xa0This\xa0momentous\xa0decree\xa0came\xa0as\xa0a\xa0great\xa0beacon\xa0light\xa0of\xa0hope\xa0to\xa0millions\xa0of\xa0Negro\xa0slaves

In [3]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

final_documents=RecursiveCharacterTextSplitter(chunk_size=2000,chunk_overlap=100).split_documents(docs)
final_documents

[Document(metadata={'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2013-01-16T16:57:22-05:00', 'author': 'Peter', 'moddate': '2014-09-29T14:17:47-04:00', 'title': 'Microsoft Word - The Most Dangerous Game', 'source': 'I Have a Dream by Martin Luther King Jr.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='I\xa0HAVE\xa0A\xa0DREAM\xa0\xa0\nMartin\xa0Luther\xa0King,\xa0Jr.\xa0\n\xa0\nI\xa0am\xa0happy\xa0to\xa0join\xa0with\xa0you\xa0today\xa0in\xa0what\xa0will\xa0go\xa0down\xa0in\xa0history\xa0as\xa0the\xa0greatest\xa0demonstration\xa0for\xa0freedom\xa0in\xa0the\xa0\nhistory\xa0of\xa0our\xa0nation.\xa0\n\xa0\nFive\xa0score\xa0years\xa0ago,\xa0a\xa0great\xa0American,\xa0in\xa0whose\xa0symbolic\xa0shadow\xa0we\xa0stand\xa0today,\xa0signed\xa0the\xa0Emancipation\xa0\nProclamation.\xa0This\xa0momentous\xa0decree\xa0came\xa0as\xa0a\xa0great\xa0beacon\xa0light\xa0of\xa0hope\xa0to\xa0millions\xa0of\xa0Negro\xa0slaves

In [4]:
len(final_documents)

7

In [5]:
from langchain_core.prompts import PromptTemplate

chunks_prompt="""
Please summarize the below speech:
Speech:`{text}'
Summary:
"""
map_prompt_template=PromptTemplate(input_variables=['text'],
                                    template=chunks_prompt)

In [6]:
final_prompt='''
Provide the final summary of the entire speech with these important points.
Add a Motivation Title,Start the precise summary with an introduction and provide the summary in number 
points for the speech.
Speech:{text}

'''
final_prompt_template=PromptTemplate(input_variables=['text'],template=final_prompt)
final_prompt_template

PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='\nProvide the final summary of the entire speech with these important points.\nAdd a Motivation Title,Start the precise summary with an introduction and provide the summary in number \npoints for the speech.\nSpeech:{text}\n\n')

In [7]:
from langchain_classic.chains.summarize import load_summarize_chain

summary_chain=load_summarize_chain(
    llm=llm,
    chain_type="map_reduce",
    map_prompt=map_prompt_template,
    combine_prompt=final_prompt_template,
    verbose=True
)

output=summary_chain.run(final_documents)
output

/tmp/ipykernel_16030/2580590070.py:11: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  output=summary_chain.run(final_documents)




> Entering new MapReduceDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

Please summarize the below speech:
Speech:`I HAVE A DREAM  
Martin Luther King, Jr. 
 
I am happy to join with you today in what will go down in history as the greatest demonstration for freedom in the 
history of our nation. 
 
Five score years ago, a great American, in whose symbolic shadow we stand today, signed the Emancipation 
Proclamation. This momentous decree came as a great beacon light of hope to millions of Negro slaves who had been 
seared in the flames of withering injustice. It came as a joyous daybreak to end the long night of their captivity. 
 
But one hundred years later, the Negro still is not free. One hundred years later, the life of the Negro is still sadly 
crippled by the manacles of segregation and the chains of discrimination. One hundred years later, the Negro lives on a 
lonely island of poverty in the midst of a vast ocean of material prosperity. 

/home/anas/LEARNING/GEN-AI/LangChain/venv/lib/python3.14/site-packages/langchain_core/language_models/base.py:463: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))




> Entering new LLMChain chain...
Prompt after formatting:

Provide the final summary of the entire speech with these important points.
Add a Motivation Title,Start the precise summary with an introduction and provide the summary in number 
points for the speech.
Speech:**Summary of Martin Luther King Jr.’s “I Have a Dream” Speech**

In his historic address, Dr. Martin Luther King Jr. celebrates the moment as a pivotal act of freedom while reminding listeners that the promise of the Emancipation Proclamation and the founding documents of the United States still remain unfulfilled for African Americans. He contrasts the hopeful dawn of emancipation with a century later in which black people continue to face segregation, discrimination, poverty, and social exclusion.

King frames the struggle as a “check” written by the nation’s founders to all citizens, guaranteeing the “unalienable Rights” of life, liberty, and the pursuit of happiness. He argues that America has defaulted on this pro

'**Motivation Title:** *Dreaming Together: The Call for Freedom, Justice, and Unity*\n\n**Introduction**  \nIn his landmark “I Have a Dream” address, Dr. Martin\u202fLuther\u202fKing\u202fJr. transforms a historical moment into a clarion call for an America that truly honors its founding promises. He urges the nation to move beyond half‑hearted reform, to confront injustice with moral strength, and to build a brotherhood that transcends race, color, and creed.\n\n**Numbered Summary**\n\n1. **Unfulfilled Promises of Freedom** – King reminds listeners that the Emancipation Proclamation and the U.S. founding documents still leave African Americans denied the full rights of life, liberty, and the pursuit of happiness.  \n2. **The “Check” Analogy** – He frames the struggle as a bad check written by the nation’s founders to people of color, asserting that the “bank of justice” is not bankrupt and that America still has the capacity to honor its obligations.  \n3. **Urgency of Immediate Actio